# WLASL RGB MobileNet-LSTM Training Notebook

This notebook trains a PyTorch `.pth` model using WLASL raw RGB videos.

It replaces the separate `mn_config.py`, `mn_dataset.py`, `mn_model.py`, and `mn_training.py` files. You only need to run this notebook in Google Colab.

Output files:
- `rgb_mobilenet_lstm.pth`
- `rgb_label_map.json`

Recommended first test:
- `MAX_CLASSES = 20`
- `NUM_FRAMES = 16`
- `BATCH_SIZE = 1`
- `EPOCHS = 1`

After it works, increase these values gradually.


In [1]:
!pip install -q kagglehub

In [2]:
import kagglehub

path = kagglehub.dataset_download("risangbaskoro/wlasl-processed")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'wlasl-processed' dataset.
Path to dataset files: /kaggle/input/wlasl-processed


## 1. Mount Google Drive

Run this cell first if your dataset is stored in Google Drive.


In [3]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
import os

print(os.listdir(path))

['nslt_2000.json', 'videos', 'nslt_1000.json', 'WLASL_v0.3.json', 'wlasl_class_list.txt', 'nslt_300.json', 'missing.txt', 'nslt_100.json']


## 2. Check GPU

In Colab, go to:

`Runtime → Change runtime type → Hardware accelerator → GPU`

Then run this cell.


In [5]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")


CUDA available: True
GPU: Tesla T4


## 3. Install / import required libraries

Colab usually already has PyTorch and torchvision. This cell installs OpenCV if needed.


In [6]:
!pip install -q opencv-python


In [7]:
import os
import cv2
import json
import random
import numpy as np
from pathlib import Path
from PIL import Image

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms


## 4. Configuration

Update `VIDEO_DIR` and `METADATA_PATH` based on your Google Drive location.

Example Drive structure:

```text
/content/drive/MyDrive/FYP sign language translator/WLASL Dataset/
    WLASL_v0.3.json
    videos/
        00635.mp4
        00636.mp4
```

For first testing, keep the settings small.


In [8]:
# ======================
# CONFIG
# ======================

VIDEO_DIR = os.path.join(path, "videos")
METADATA_PATH = os.path.join(path, "WLASL_v0.3.json")

SAVE_DIR = "/content/drive/MyDrive/FIT3164/mn"
os.makedirs(SAVE_DIR, exist_ok=True)

SAVE_MODEL_PATH = os.path.join(SAVE_DIR, "rgb_mobilenet_lstm.pth")
SAVE_LABEL_PATH = os.path.join(SAVE_DIR, "rgb_label_map.json")

print("SAVE_DIR exists:", os.path.exists(SAVE_DIR))
print("SAVE_MODEL_PATH:", SAVE_MODEL_PATH)

NUM_FRAMES = 16
IMG_SIZE = 224
BATCH_SIZE = 1
EPOCHS = 1
LR = 1e-4

# Start small first. Later you can try 50, 100, 500, or None for all classes.
MAX_CLASSES = 20

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", DEVICE)
print("VIDEO_DIR exists:", os.path.exists(VIDEO_DIR))
print("METADATA_PATH exists:", os.path.exists(METADATA_PATH))


SAVE_DIR exists: True
SAVE_MODEL_PATH: /content/drive/MyDrive/FIT3164/mn/rgb_mobilenet_lstm.pth
Device: cuda
VIDEO_DIR exists: True
METADATA_PATH exists: True


In [9]:
for root, dirs, files in os.walk(path):
    print("ROOT:", root)
    print("DIRS:", dirs[:10])
    print("FILES:", files[:10])
    print()
    break

ROOT: /kaggle/input/wlasl-processed
DIRS: ['videos']
FILES: ['nslt_2000.json', 'nslt_1000.json', 'WLASL_v0.3.json', 'wlasl_class_list.txt', 'nslt_300.json', 'missing.txt', 'nslt_100.json']



## 5. Optional: copy videos to Colab local storage for faster training

Reading many videos directly from Google Drive can be slow.

If your videos are already unzipped in Drive, you can skip this section.

If you have a `videos.zip`, use this idea:

```python
!cp "/content/drive/MyDrive/.../videos.zip" "/content/videos.zip"
!unzip -q "/content/videos.zip" -d "/content/WLASL_Dataset"
VIDEO_DIR = "/content/WLASL_Dataset/videos"
```


In [10]:
# OPTIONAL FAST MODE
# Uncomment and edit this only if you have a videos.zip file in Google Drive.

# !cp "/content/drive/MyDrive/FYP sign language translator/WLASL Dataset/videos.zip" "/content/videos.zip"
# !unzip -q "/content/videos.zip" -d "/content/WLASL_Dataset"

# VIDEO_DIR = "/content/WLASL_Dataset/videos"
# print("Updated VIDEO_DIR:", VIDEO_DIR)
# print("VIDEO_DIR exists:", os.path.exists(VIDEO_DIR))


## 6. Build label map from WLASL metadata

This reads `WLASL_v0.3.json` and creates:
- `label2id`
- `id2label`

The first test uses only the first `MAX_CLASSES` glosses.


In [11]:
def build_label_map(metadata_path, max_classes=None):
    with open(metadata_path, "r", encoding="utf-8") as f:
        metadata = json.load(f)

    labels = sorted([item["gloss"] for item in metadata])

    if max_classes is not None:
        labels = labels[:max_classes]

    label2id = {label: i for i, label in enumerate(labels)}
    id2label = {i: label for label, i in label2id.items()}

    return label2id, id2label


label2id, id2label = build_label_map(METADATA_PATH, MAX_CLASSES)
num_classes = len(label2id)

print("Number of classes:", num_classes)
print("Classes:", label2id)

with open(SAVE_LABEL_PATH, "w", encoding="utf-8") as f:
    json.dump(
        {
            "label2id": label2id,
            "id2label": id2label
        },
        f,
        indent=2
    )

print("Saved label map to:", SAVE_LABEL_PATH)


Number of classes: 20
Classes: {'a': 0, 'a lot': 1, 'abdomen': 2, 'able': 3, 'about': 4, 'above': 5, 'accent': 6, 'accept': 7, 'accident': 8, 'accomplish': 9, 'accountant': 10, 'across': 11, 'act': 12, 'action': 13, 'active': 14, 'activity': 15, 'actor': 16, 'adapt': 17, 'add': 18, 'address': 19}
Saved label map to: /content/drive/MyDrive/FIT3164/mn/rgb_label_map.json


## 7. Dataset class

This dataset:
1. Reads WLASL JSON metadata.
2. Filters by split: `train`, `val`, or `test`.
3. Finds matching videos in `VIDEO_DIR`.
4. Samples a fixed number of frames.
5. Converts frames into tensors.


In [12]:
class WLASLVideoDataset(Dataset):
    def __init__(self, video_dir, metadata_path, split, label2id, transform=None):
        self.video_dir = Path(video_dir)
        self.metadata_path = metadata_path
        self.split = split
        self.label2id = label2id
        self.transform = transform
        self.samples = []

        with open(metadata_path, "r", encoding="utf-8") as f:
            metadata = json.load(f)

        missing_count = 0

        for item in metadata:
            gloss = item["gloss"]

            if gloss not in label2id:
                continue

            for inst in item["instances"]:
                if inst.get("split") != split:
                    continue

                video_id = inst.get("video_id")
                video_path = self.find_video_file(video_id)

                if video_path is not None:
                    self.samples.append((str(video_path), label2id[gloss], gloss, video_id))
                else:
                    missing_count += 1

        print(f"{split} samples loaded:", len(self.samples))
        print(f"{split} missing videos:", missing_count)

    def find_video_file(self, video_id):
        possible_exts = [".mp4", ".avi", ".mov", ".webm", ".mkv"]

        for ext in possible_exts:
            path = self.video_dir / f"{video_id}{ext}"
            if path.exists():
                return path

        return None

    def __len__(self):
        return len(self.samples)

    def sample_frames(self, video_path):
        cap = cv2.VideoCapture(video_path)
        frames = []

        while True:
            ret, frame = cap.read()

            if not ret:
                break

            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frames.append(frame)

        cap.release()

        if len(frames) == 0:
            frames = [np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)]

        if len(frames) >= NUM_FRAMES:
            indices = np.linspace(0, len(frames) - 1, NUM_FRAMES).astype(int)
            frames = [frames[i] for i in indices]
        else:
            last = frames[-1]
            while len(frames) < NUM_FRAMES:
                frames.append(last)

        processed = []

        for frame in frames:
            img = Image.fromarray(frame)

            if self.transform:
                img = self.transform(img)

            processed.append(img)

        return torch.stack(processed)

    def __getitem__(self, idx):
        video_path, label, gloss, video_id = self.samples[idx]
        frames = self.sample_frames(video_path)

        return frames, torch.tensor(label, dtype=torch.long)


## 8. Create transforms and datasets

Expected batch shape:

```text
[BATCH_SIZE, NUM_FRAMES, 3, 224, 224]
```

Example:

```text
torch.Size([1, 16, 3, 224, 224])
```


In [13]:
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.0),  # Keep 0.0 for sign language because flipping may change meaning.
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

train_dataset = WLASLVideoDataset(
    video_dir=VIDEO_DIR,
    metadata_path=METADATA_PATH,
    split="train",
    label2id=label2id,
    transform=train_transform
)

val_dataset = WLASLVideoDataset(
    video_dir=VIDEO_DIR,
    metadata_path=METADATA_PATH,
    split="val",
    label2id=label2id,
    transform=val_transform
)

print("Train samples:", len(train_dataset))
print("Val samples:", len(val_dataset))

if len(train_dataset) == 0:
    raise ValueError("No training samples found. Check VIDEO_DIR, METADATA_PATH, video filenames, and split names.")


train samples loaded: 83
train missing videos: 41
val samples loaded: 25
val missing videos: 14
Train samples: 83
Val samples: 25


In [14]:
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

frames, labels = next(iter(train_loader))
print("Debug batch frames shape:", frames.shape)
print("Debug batch labels shape:", labels.shape)
print("Labels:", labels)


Debug batch frames shape: torch.Size([1, 16, 3, 224, 224])
Debug batch labels shape: torch.Size([1])
Labels: tensor([11])


## 9. Define MobileNetV2 + LSTM model

The model works like this:

```text
Video frames
    ↓
MobileNetV2 CNN feature extractor
    ↓
LSTM sequence model
    ↓
Classifier
    ↓
Gloss prediction
```

For easier training and lower GPU memory use, MobileNetV2 is frozen first.


In [15]:
class MobileNetLSTM(nn.Module):
    def __init__(self, num_classes, hidden_size=256, num_layers=1, freeze_cnn=True):
        super().__init__()

        mobilenet = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.DEFAULT)
        self.cnn = mobilenet.features

        if freeze_cnn:
            for param in self.cnn.parameters():
                param.requires_grad = False

        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.feature_dim = 1280

        self.lstm = nn.LSTM(
            input_size=self.feature_dim,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True
        )

        self.classifier = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(hidden_size * 2, num_classes)
        )

    def forward(self, x):
        # x shape: B, T, C, H, W
        B, T, C, H, W = x.shape

        # CNN expects image batch: B*T, C, H, W
        x = x.view(B * T, C, H, W)

        feat = self.cnn(x)
        feat = self.pool(feat)

        # Back to sequence shape: B, T, feature_dim
        feat = feat.view(B, T, self.feature_dim)

        lstm_out, _ = self.lstm(feat)

        # Use final time step
        final_feat = lstm_out[:, -1, :]

        logits = self.classifier(final_feat)
        return logits


model = MobileNetLSTM(num_classes=num_classes, freeze_cnn=True).to(DEVICE)

print(model.__class__.__name__)
print("Model ready.")


MobileNetLSTM
Model ready.


## 10. Test one forward pass

Run this before training to make sure the model can process one batch.


In [16]:
if torch.cuda.is_available():
    torch.cuda.empty_cache()

model.eval()

with torch.no_grad():
    frames = frames.to(DEVICE)
    logits = model(frames)

print("Logits shape:", logits.shape)
print("Expected:", (BATCH_SIZE, num_classes))


Logits shape: torch.Size([1, 20])
Expected: (1, 20)


## 11. Training loop

This cell trains the model and saves the best validation model as `.pth`.

If Colab runs out of memory:
- set `BATCH_SIZE = 1`
- set `NUM_FRAMES = 16`
- keep `freeze_cnn=True`


In [17]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LR
)

best_val_acc = 0.0

if torch.cuda.is_available():
    torch.cuda.empty_cache()

for epoch in range(EPOCHS):
    model.train()

    total_loss = 0.0
    correct = 0
    total = 0

    for batch_idx, (frames, labels) in enumerate(train_loader):
        frames = frames.to(DEVICE)
        labels = labels.to(DEVICE)

        optimizer.zero_grad()

        logits = model(frames)
        loss = criterion(logits, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        preds = logits.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

        if (batch_idx + 1) % 10 == 0:
            print(
                f"Epoch {epoch + 1}/{EPOCHS} | "
                f"Batch {batch_idx + 1}/{len(train_loader)} | "
                f"Loss: {loss.item():.4f}"
            )

    train_acc = correct / total if total > 0 else 0

    model.eval()

    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for frames, labels in val_loader:
            frames = frames.to(DEVICE)
            labels = labels.to(DEVICE)

            logits = model(frames)
            preds = logits.argmax(dim=1)

            val_correct += (preds == labels).sum().item()
            val_total += labels.size(0)

    val_acc = val_correct / val_total if val_total > 0 else 0

    print(
        f"Epoch [{epoch + 1}/{EPOCHS}] "
        f"Loss: {total_loss:.4f} "
        f"Train Acc: {train_acc:.4f} "
        f"Val Acc: {val_acc:.4f}"
    )

    if val_acc >= best_val_acc:
        best_val_acc = val_acc

        torch.save(
            {
                "model_state_dict": model.state_dict(),
                "num_classes": num_classes,
                "label2id": label2id,
                "id2label": id2label,
                "num_frames": NUM_FRAMES,
                "img_size": IMG_SIZE,
                "max_classes": MAX_CLASSES,
                "model_type": "MobileNetV2_LSTM_RGB",
                "freeze_cnn": True
            },
            SAVE_MODEL_PATH
        )

        print("Saved best model to:", SAVE_MODEL_PATH)

print("Training complete.")
print("Best Val Acc:", best_val_acc)


Epoch 1/1 | Batch 10/83 | Loss: 3.0134
Epoch 1/1 | Batch 20/83 | Loss: 3.0683
Epoch 1/1 | Batch 30/83 | Loss: 2.9789
Epoch 1/1 | Batch 40/83 | Loss: 2.8643
Epoch 1/1 | Batch 50/83 | Loss: 3.1737
Epoch 1/1 | Batch 60/83 | Loss: 2.7599
Epoch 1/1 | Batch 70/83 | Loss: 2.8228
Epoch 1/1 | Batch 80/83 | Loss: 3.1831
Epoch [1/1] Loss: 252.1501 Train Acc: 0.0361 Val Acc: 0.0000
Saved best model to: /content/drive/MyDrive/FIT3164/mn/rgb_mobilenet_lstm.pth
Training complete.
Best Val Acc: 0.0


## 12. Check saved files

Run this to confirm the `.pth` and label map were created.


In [18]:
print("Model exists:", os.path.exists(SAVE_MODEL_PATH))
print("Label map exists:", os.path.exists(SAVE_LABEL_PATH))

print("Model path:", SAVE_MODEL_PATH)
print("Label map path:", SAVE_LABEL_PATH)

if os.path.exists(SAVE_MODEL_PATH):
    print("Model size MB:", os.path.getsize(SAVE_MODEL_PATH) / (1024 * 1024))


Model exists: True
Label map exists: True
Model path: /content/drive/MyDrive/FIT3164/mn/rgb_mobilenet_lstm.pth
Label map path: /content/drive/MyDrive/FIT3164/mn/rgb_label_map.json
Model size MB: 20.774250030517578


## 13. Load saved `.pth` for verification

This checks that the saved checkpoint can be loaded again.


In [19]:
checkpoint = torch.load(SAVE_MODEL_PATH, map_location=DEVICE)

print("Checkpoint keys:", checkpoint.keys())
print("num_classes:", checkpoint["num_classes"])
print("num_frames:", checkpoint["num_frames"])
print("img_size:", checkpoint["img_size"])
print("model_type:", checkpoint["model_type"])


Checkpoint keys: dict_keys(['model_state_dict', 'num_classes', 'label2id', 'id2label', 'num_frames', 'img_size', 'max_classes', 'model_type', 'freeze_cnn'])
num_classes: 20
num_frames: 16
img_size: 224
model_type: MobileNetV2_LSTM_RGB


## 14. Next training adjustments

Once the notebook successfully generates `.pth`, you can try stronger settings:

Small:
```python
MAX_CLASSES = 50
EPOCHS = 3
NUM_FRAMES = 16
BATCH_SIZE = 1
```

Medium:
```python
MAX_CLASSES = 100
EPOCHS = 5
NUM_FRAMES = 30
BATCH_SIZE = 1
```

Do not directly jump to all 2000 WLASL classes until the pipeline is stable.
